# Cut text-to-SQL prompt tokens with schema selection

A text-to-SQL prompt carries the database schema. Send all of it and every question pays for every
table, on every call. On a real warehouse that is tens of thousands of tokens, most of them about
tables the question never touches.

This notebook sends Claude only the tables a question needs, picked by
[schemagate](https://github.com/ashishsinha1602/schemagate) (an open-source, offline selector:
BM25 plus a small embedder, no model call). It measures the difference with the token-counting
API, then writes and runs the SQL.

What you will see:

1. the full schema against the selected schema, in input tokens;
2. Claude answering from the selection, with the SQL run against a real SQLite database;
3. the selection respecting who is asking: a table the caller may not read never enters the prompt;
4. what the saving is worth per month, and when it is not worth much.

In [1]:
%pip install -q "schemagate>=1.14" anthropic python-dotenv

In [2]:
import os
import sqlite3

import anthropic
from dotenv import load_dotenv
from schemagate import Principal
from schemagate.demo_schema import demo_catalog

load_dotenv()
client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
MODEL = "claude-haiku-4-5"

## 1. A schema with more tables than any question needs

The bundled demo schema has 42 tables and views across a commerce, billing, HR and inventory
database, with the traps real schemas have: near-duplicate tables, cryptic legacy names, join tables
a question never mentions. It is materialised as a SQLite file with a few rows, so the SQL can run.

In [3]:
cat = demo_catalog()
objects = list(cat.objects())
full_schema = "\n\n".join(o.render_ddl(max_columns=10**6) for o in objects)
print(len(objects), "objects,", len(full_schema), "characters of DDL")

42 objects, 8823 characters of DDL


## 2. Full schema against selection, in real input tokens

`messages.count_tokens` reports what a request would cost without running it. The prompt is the
same in both cases except for the schema block.

In [4]:
QUESTION = "Which customers owe us money, and how much?"


def prompt(schema: str, question: str) -> str:
    return (
        f"Database schema (SQLite):\n\n{schema}\n\nQuestion: {question}\n\n"
        "Reply with one SQLite SELECT and nothing else."
    )


def input_tokens(text: str) -> int:
    return client.messages.count_tokens(
        model=MODEL, messages=[{"role": "user", "content": text}]
    ).input_tokens


finance = Principal("okta:analyst", roles={"finance"})
selection = cat.select(QUESTION, top_k=6, principal=finance)
selected_schema = selection.prompt_fragment()

full = input_tokens(prompt(full_schema, QUESTION))
small = input_tokens(prompt(selected_schema, QUESTION))
print("tables sent:", [t.split(".")[-1] for t in selection.table_names])
print(f"full schema: {full:,} input tokens")
print(f"selection:   {small:,} input tokens  ({1 - small / full:.0%} fewer)")

tables sent: ['crm_customer', 'v_customer_balance', 'crm_contact', 'crm_opportunity', 'sales_order', 'sales_quote', 'core_party', 'core_address', 'sales_order_status']
full schema: 3,073 input tokens
selection:   772 input tokens  (75% fewer)


## 3. Claude answers from the selection

The SQL runs against the demo database, so you can check the answer is real, not just plausible.

In [5]:
def ask(schema: str, question: str) -> str:
    try:
        msg = client.messages.create(
            model=MODEL,
            max_tokens=400,
            messages=[{"role": "user", "content": prompt(schema, question)}],
        )
    except anthropic.APIError as e:
        raise SystemExit(f"API call failed: {e}") from e
    sql = msg.content[0].text.strip()
    return sql.removeprefix("```sql").removeprefix("```").removesuffix("```").strip()


sql = ask(selected_schema, QUESTION)
print(sql)

db = sqlite3.connect(cat._demo_url.removeprefix("sqlite:///"))
rows = db.execute(sql).fetchall()
print(len(rows), "rows, first five:")
for row in rows[:5]:
    print(tuple(round(v, 2) if isinstance(v, float) else v for v in row))

SELECT 
  c.id,
  c.account_number,
  (COALESCE(vcb.invoiced, 0) - COALESCE(vcb.paid, 0)) AS amount_owed
FROM crm_customer c
JOIN v_customer_balance vcb ON c.id = vcb.id_customer
WHERE (COALESCE(vcb.invoiced, 0) - COALESCE(vcb.paid, 0)) > 0
ORDER BY amount_owed DESC
17 rows, first five:
(17, 'AC-01119', 4719.57)
(13, 'AC-01091', 3312.36)
(14, 'AC-01098', 3096.73)
(18, 'AC-01126', 2821.75)
(2, 'AC-01014', 2784.9)


## 4. Selection is also access control

`hr_compensation` is readable only by the `payroll` role. For anyone else it is not ranked low; it is
absent, so the model cannot write a query against it or mention that it exists.

In [6]:
SALARY = "salary and pay grade by employee"
for who in (Principal("okta:analyst", roles={"finance"}), Principal("okta:hr", roles={"payroll"})):
    tables = [t.split(".")[-1] for t in cat.select(SALARY, top_k=6, principal=who).table_names]
    print(sorted(who.roles), "->", "hr_compensation" in tables, tables)

['finance'] -> False ['hr_employee', 'v_employee_headcount', 'sales_order_status', 'v_monthly_revenue', 'billing_tax_rate', 'core_party', 'hr_department', 'core_country', 'core_address']
['payroll'] -> True ['hr_compensation', 'hr_employee', 'v_employee_headcount', 'sales_order_status', 'v_monthly_revenue', 'billing_tax_rate', 'core_party', 'hr_department', 'core_country']


## 5. What it is worth, and when it is not

Replace the volume and the price with your own. The selection stays at a handful of tables (the top six, plus the tables they
join to) however large the schema grows, so the saving grows with the schema.

In [7]:
questions_per_month = 1_000_000
price_per_million_input = 1.00  # USD, put your model's input price here


def monthly(tokens_per_question: int) -> float:
    return tokens_per_question * questions_per_month / 1e6 * price_per_million_input


print(f"full schema: ${monthly(full):,.0f}/month")
print(f"selection:   ${monthly(small):,.0f}/month")

full schema: $3,073/month
selection:   $772/month


On public benchmarks, measured the same way (no model called; the script and per-database numbers
are in schemagate's
[BENCHMARKS.md](https://github.com/ashishsinha1602/schemagate/blob/main/BENCHMARKS.md#prompt-tokens-on-public-schemas-the-saving-grows-with-the-schema)):

| Schemas | Median tables | Median saving per database |
|---|---|---|
| BIRD dev | 7 | 8.8% |
| Spider 2.0-lite BigQuery and Snowflake | 16 | 48.8% |
| the same, databases with 50+ tables | | 92.9% |

A seven-table database is already a few hundred tokens, so there is little to cut. Selection pays off
on big schemas. It also has a price: a table the question needs can be left out (on the Spider 2.0
warehouses, every gold table was selected for 79.6% of questions), so measure recall on your own
questions before relying on it.